# Chapter 4.2: Prefix Caching — Automatic Prefix Caching (APC)

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** why prefix caching is critical for production LLM serving
2. **Explain** how Automatic Prefix Caching (APC) uses hash-based block matching
3. **Trace** through vLLM's APC source code: content-addressable blocks
4. **Analyze** eviction policies (LRU) and their tradeoffs
5. **Measure** cache hit rates and performance improvements
6. **Design** optimal caching strategies for specific workloads

---

## Prerequisites
- Chapter 4.1 (KV-Cache Memory Layout)
- Understanding of hash tables and LRU caches
- Familiarity with multi-turn conversation patterns

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part4/chapter_4.2_prefix_caching.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part4/chapter_4.2_prefix_caching.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Prefix Caching Matters

### The Redundancy Problem

In production LLM serving, many requests share **identical prefixes**:

```
Pattern 1: System Prompts
  All requests to a chatbot share the same system prompt:
  "You are a helpful assistant. You should..." (500+ tokens)
  
  Without caching: 100 concurrent requests × 500 tokens = 50,000 redundant tokens
  With caching: 500 tokens cached once, reused 100 times

Pattern 2: Multi-turn Conversations
  Turn 1: [System] [User Q1] [Assistant A1]
  Turn 2: [System] [User Q1] [Assistant A1] [User Q2] [Assistant A2]
  Turn 3: [System] [User Q1] [Assistant A1] [User Q2] [Assistant A2] [User Q3]
  
  Each turn recomputes ALL previous context without caching!

Pattern 3: Few-shot Examples
  Many applications use the same few-shot examples for all requests:
  [Examples: 1000 tokens] [New query: 50 tokens]
  
  Without caching: 95% of computation is wasted on repeated examples
```

### Performance Impact

| Scenario | Without APC | With APC | Speedup |
|----------|-------------|----------|----------|
| System prompt (500 tok) | 500 tok prefill | ~0 tok prefill | ~10x TTFT |
| Multi-turn (2000 tok context) | 2000 tok prefill | ~50 tok prefill | ~40x TTFT |
| Few-shot (1000 tok examples) | 1000 tok prefill | ~0 tok prefill | ~20x TTFT |

In [ ]:
import hashlib
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple, Set
from collections import OrderedDict
from dataclasses import dataclass, field

print("Imports loaded successfully.")

## 2. How APC Works: Hash-Based Block Matching

### Core Mechanism

APC treats KV-cache blocks as **content-addressable storage**. Each block is identified by a hash of its content (token IDs), not by which sequence owns it.

```
Traditional Block Allocation:
  Sequence 0 → Block Table: [Block_A, Block_B, Block_C]
  Sequence 1 → Block Table: [Block_D, Block_E, Block_F]  ← Even if same content!

APC Block Allocation:
  Hash(tokens[0:16])  = 0xABCD → Block_A
  Hash(tokens[16:32]) = 0xEF01 → Block_B  
  
  Sequence 0 → Block Table: [Block_A, Block_B, Block_C]
  Sequence 1 → Block Table: [Block_A, Block_B, Block_F]  ← Shares prefix blocks!
```

### Hash Function Design

The hash for a block depends on ALL preceding tokens, not just the tokens in the block. This ensures that blocks with the same tokens but different preceding context get different hashes:

```python
# From vllm/core/block/prefix_caching_block.py
# The hash is computed as:
#   hash(block_i) = hash(token_ids[0 : (i+1)*block_size])
# This creates a chain: each block's identity depends on its full prefix.
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Figure: Automatic Prefix Caching (APC) Diagram ──
BLUE = '#4A90D9'; GREEN = '#27AE60'; ORANGE = '#F39C12'; RED = '#E74C3C'; PURPLE = '#8E44AD'

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16); ax.set_ylim(0, 9); ax.axis('off')
fig.patch.set_facecolor('white')

# System prompt blocks (shared)
ax.text(8, 8.5, 'Shared System Prompt KV Blocks (hash-matched)',
        fontsize=13, fontweight='bold', ha='center', color=ORANGE)

prompt_blocks_x = [3.5, 6.5, 9.5]
for i, x in enumerate(prompt_blocks_x):
    rect = mpatches.FancyBboxPatch((x-1, 7.0), 2.0, 1.0,
        boxstyle="round,pad=0.1", facecolor=ORANGE, edgecolor='white', linewidth=2.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, 7.5, f'Sys Block {i}\nhash=0x{1234+i:04X}\nrc=3',
            ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# Three requests, each with their own user query blocks
requests = [
    ('Request 0', 2.5,  4.5, BLUE,   'User Q0'),
    ('Request 1', 7.0,  4.5, GREEN,  'User Q1'),
    ('Request 2', 11.5, 4.5, PURPLE, 'User Q2'),
]

for name, x, y, color, qlabel in requests:
    # Request label
    ax.text(x, y + 1.5, name, fontsize=10, fontweight='bold', ha='center', color=color)
    # User query block
    rect = mpatches.FancyBboxPatch((x-1, y-0.5), 2.0, 1.0,
        boxstyle="round,pad=0.1", facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x, y, f'{qlabel}\n(unique)', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')
    # Arrows from request to shared prompt blocks
    for px in prompt_blocks_x:
        ax.annotate('', xy=(px, 7.0), xytext=(x, y + 1.2),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.2, alpha=0.5,
                                   connectionstyle='arc3,rad=0.1'))

# Bottom summary
ax.add_patch(mpatches.FancyBboxPatch((2, 1.0), 12, 1.8,
    boxstyle="round,pad=0.2", facecolor='lightyellow', edgecolor=ORANGE, linewidth=2))
ax.text(8, 2.2, 'Without APC: 3 copies of system prompt = 9 blocks',
        fontsize=10, ha='center', color=RED, fontweight='bold')
ax.text(8, 1.5, 'With APC: 1 shared copy + 3 unique query blocks = 6 blocks (33% savings)',
        fontsize=10, ha='center', color=GREEN, fontweight='bold')

ax.set_title('Automatic Prefix Caching: Multiple Requests Share System Prompt KV Blocks',
             fontsize=14, fontweight='bold', pad=10)
plt.tight_layout(); plt.show()

In [ ]:
class ContentHash:
    """
    Computes content-addressable hashes for KV-cache blocks.
    
    Mirrors the hashing logic in vllm/core/block/prefix_caching_block.py.
    Key insight: the hash of block i includes ALL preceding tokens,
    ensuring that context-dependent blocks have unique identities.
    """
    
    @staticmethod
    def hash_block(token_ids: Tuple[int, ...], 
                   block_size: int, 
                   block_idx: int) -> Optional[int]:
        """
        Compute hash for block at given index.
        
        Only full blocks can be hashed (partial blocks might change).
        
        Args:
            token_ids: All token IDs up to this point
            block_size: Number of tokens per block
            block_idx: Which block to hash
        
        Returns:
            Hash value, or None if block is not full
        """
        end_idx = (block_idx + 1) * block_size
        
        # Only hash full blocks
        if end_idx > len(token_ids):
            return None
        
        # Hash ALL tokens up to and including this block
        prefix = token_ids[:end_idx]
        return hash(prefix)
    
    @staticmethod
    def compute_all_hashes(token_ids: Tuple[int, ...], 
                            block_size: int) -> List[Optional[int]]:
        """Compute hashes for all blocks in a sequence."""
        num_blocks = (len(token_ids) + block_size - 1) // block_size
        hashes = []
        for i in range(num_blocks):
            h = ContentHash.hash_block(token_ids, block_size, i)
            hashes.append(h)
        return hashes

# Demonstrate hashing
block_size = 4  # Small for demo

# Two sequences sharing a prefix
seq1 = tuple([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
seq2 = tuple([1, 2, 3, 4, 5, 6, 7, 8, 20, 21, 22, 23])  # Diverges at token 9

hashes1 = ContentHash.compute_all_hashes(seq1, block_size)
hashes2 = ContentHash.compute_all_hashes(seq2, block_size)

print("Hash comparison (block_size=4):")
print(f"{'Block':>6} {'Seq1 Tokens':>20} {'Seq2 Tokens':>20} {'Match':>6}")
print("-" * 58)
for i in range(3):
    s = i * block_size
    e = s + block_size
    match = "YES" if hashes1[i] == hashes2[i] else "NO"
    print(f"{i:>6} {str(seq1[s:e]):>20} {str(seq2[s:e]):>20} {match:>6}")

print(f"\nBlocks 0-1 share identical prefix, so their hashes match.")
print(f"Block 2 differs because the content diverges.")

In [ ]:
@dataclass
class CachedBlock:
    """A block in the prefix cache."""
    block_id: int
    content_hash: int
    ref_count: int = 0
    last_access_time: float = 0.0
    is_computed: bool = True  # Whether KV values have been computed


class PrefixCachingBlockAllocator:
    """
    Block allocator with Automatic Prefix Caching (APC).
    
    Mirrors: vllm/core/block/prefix_caching_block.py
    
    Key concepts:
    1. Content-addressable: blocks identified by hash of content
    2. Cached blocks: computed blocks kept even after sequence completes
    3. LRU eviction: least recently used cached blocks evicted first
    4. Cache hit: reuse cached block instead of recomputing
    """
    
    def __init__(self, num_blocks: int, block_size: int = 16):
        self.num_blocks = num_blocks
        self.block_size = block_size
        
        # All blocks start as free
        self.free_blocks: List[int] = list(range(num_blocks))
        
        # Hash -> CachedBlock mapping (content-addressable lookup)
        self.hash_to_block: Dict[int, CachedBlock] = {}
        
        # Eviction order: LRU cache of unreferenced blocks
        # These are computed but not currently in use by any sequence
        self.evictor: OrderedDict[int, CachedBlock] = OrderedDict()
        
        # Active blocks (referenced by at least one sequence)
        self.active_blocks: Dict[int, CachedBlock] = {}
        
        # Metrics
        self.total_lookups = 0
        self.cache_hits = 0
        self.cache_misses = 0
        self.evictions = 0
        self._time = 0
    
    def _get_time(self):
        self._time += 1
        return self._time
    
    def allocate_block(self, content_hash: Optional[int] = None) -> Optional[CachedBlock]:
        """
        Allocate a block, optionally matching by content hash.
        
        Three possible outcomes:
        1. Cache HIT: block with matching hash exists and is computed
        2. Cache MISS (free): allocate from free pool
        3. Cache MISS (evict): evict LRU block and reuse
        """
        self.total_lookups += 1
        
        # Try cache hit first
        if content_hash is not None and content_hash in self.hash_to_block:
            block = self.hash_to_block[content_hash]
            self.cache_hits += 1
            
            # Move from evictor to active if it was cached
            if block.block_id in self.evictor:
                del self.evictor[block.block_id]
            
            block.ref_count += 1
            block.last_access_time = self._get_time()
            self.active_blocks[block.block_id] = block
            return block
        
        # Cache miss
        self.cache_misses += 1
        
        # Try free pool
        if self.free_blocks:
            block_id = self.free_blocks.pop()
        elif self.evictor:
            # Evict LRU block
            evicted_id, evicted_block = self.evictor.popitem(last=False)
            block_id = evicted_id
            # Remove old hash mapping
            if evicted_block.content_hash in self.hash_to_block:
                del self.hash_to_block[evicted_block.content_hash]
            self.evictions += 1
        else:
            return None  # No blocks available
        
        # Create new block
        block = CachedBlock(
            block_id=block_id,
            content_hash=content_hash if content_hash else hash(random.random()),
            ref_count=1,
            last_access_time=self._get_time(),
            is_computed=False,  # Needs computation
        )
        
        if content_hash is not None:
            self.hash_to_block[content_hash] = block
        self.active_blocks[block_id] = block
        return block
    
    def free_block(self, block: CachedBlock):
        """
        Release a reference to a block.
        
        If ref_count reaches 0, the block moves to the evictor
        (cached for potential reuse) rather than being freed immediately.
        """
        block.ref_count -= 1
        
        if block.ref_count == 0:
            # Move to evictor (keep cached for reuse)
            if block.block_id in self.active_blocks:
                del self.active_blocks[block.block_id]
            
            if block.is_computed:
                # Keep in cache for potential reuse
                self.evictor[block.block_id] = block
            else:
                # Not computed, just free it
                self.free_blocks.append(block.block_id)
                if block.content_hash in self.hash_to_block:
                    del self.hash_to_block[block.content_hash]
    
    def mark_computed(self, block: CachedBlock):
        """Mark block as having valid KV values."""
        block.is_computed = True
    
    @property
    def hit_rate(self) -> float:
        if self.total_lookups == 0:
            return 0.0
        return self.cache_hits / self.total_lookups
    
    @property
    def stats(self) -> dict:
        return {
            'total_lookups': self.total_lookups,
            'cache_hits': self.cache_hits,
            'cache_misses': self.cache_misses,
            'hit_rate': self.hit_rate,
            'evictions': self.evictions,
            'active_blocks': len(self.active_blocks),
            'cached_blocks': len(self.evictor),
            'free_blocks': len(self.free_blocks),
        }

print("PrefixCachingBlockAllocator defined.")

In [ ]:
# Demonstrate APC with system prompt sharing

allocator = PrefixCachingBlockAllocator(num_blocks=100, block_size=4)

# System prompt (shared across all requests)
system_prompt = tuple(range(1, 13))  # 12 tokens = 3 blocks

# Three requests with same system prompt but different user queries
requests = [
    tuple(list(system_prompt) + [100, 101, 102, 103]),  # User query 1
    tuple(list(system_prompt) + [200, 201, 202, 203]),  # User query 2  
    tuple(list(system_prompt) + [300, 301, 302, 303]),  # User query 3
]

print("=== APC Demo: System Prompt Sharing ===")
print(f"System prompt: {len(system_prompt)} tokens = {len(system_prompt)//4} blocks")
print()

all_blocks = []
for req_idx, tokens in enumerate(requests):
    print(f"Request {req_idx}: {len(tokens)} tokens")
    hashes = ContentHash.compute_all_hashes(tokens, block_size=4)
    req_blocks = []
    
    for blk_idx, h in enumerate(hashes):
        block = allocator.allocate_block(content_hash=h)
        if block.is_computed:
            print(f"  Block {blk_idx}: CACHE HIT (block_id={block.block_id})")
        else:
            allocator.mark_computed(block)
            print(f"  Block {blk_idx}: MISS → computed (block_id={block.block_id})")
        req_blocks.append(block)
    all_blocks.append(req_blocks)
    print()

print(f"\n=== Stats ===")
for k, v in allocator.stats.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.2%}")
    else:
        print(f"  {k}: {v}")

print(f"\nShared system prompt blocks saved {allocator.cache_hits * 4} tokens of recomputation!")

## 3. Content-Addressable Block Storage: Source Code Deep Dive

### vLLM Source: `vllm/core/block/prefix_caching_block.py`

```python
class PrefixCachingBlock(Block):
    """A block that supports prefix caching.
    
    The key insight is that blocks are identified by their content hash,
    not by their physical block ID. This enables content-addressable lookup.
    """
    
    def __init__(self, prev_block: Optional[Block], 
                 token_ids: List[int], 
                 block_size: int,
                 allocator: PrefixCachingBlockAllocator,
                 block_id: Optional[int] = None):
        self._prev_block = prev_block
        self._token_ids = token_ids
        self._block_size = block_size
        self._allocator = allocator
        self._block_id = block_id
        self._content_hash: Optional[int] = None
        
        # Compute hash when block is full
        self._update_hash()
    
    @property
    def content_hash(self) -> Optional[int]:
        return self._content_hash
    
    def _update_hash(self):
        """Compute content hash when block becomes full."""
        if len(self._token_ids) < self._block_size:
            return  # Only full blocks get hashed
        
        # Collect all previous token IDs
        previous_tokens = []
        block = self._prev_block
        while block is not None:
            previous_tokens = block.token_ids + previous_tokens
            block = block.prev_block
        
        all_tokens = tuple(previous_tokens + self._token_ids)
        self._content_hash = hash(all_tokens)
```

### Hash Chain Property

```
Block 0: hash([t0, t1, t2, t3])
Block 1: hash([t0, t1, t2, t3, t4, t5, t6, t7])
Block 2: hash([t0, t1, t2, t3, t4, t5, t6, t7, t8, t9, t10, t11])

This ensures:
  - Same tokens in same position → same hash
  - Same tokens in different position → different hash
  - Hash lookup is O(1) once computed
```

In [ ]:
# Demonstrate hash chain property

block_size = 4

# Sequence: [10, 20, 30, 40, 50, 60, 70, 80]
seq = tuple([10, 20, 30, 40, 50, 60, 70, 80])

print("Hash Chain Demo:")
print(f"Sequence: {seq}")
print()

for block_idx in range(len(seq) // block_size):
    end = (block_idx + 1) * block_size
    prefix = seq[:end]
    block_tokens = seq[block_idx * block_size : end]
    h = hash(prefix)
    print(f"Block {block_idx}:")
    print(f"  Block tokens: {block_tokens}")
    print(f"  Full prefix:  {prefix}")
    print(f"  Hash: {h}")
    print()

# Show that same tokens in different positions give different hashes
print("--- Position Sensitivity ---")
tokens_at_pos0 = tuple([50, 60, 70, 80])
tokens_at_pos1 = tuple([10, 20, 30, 40, 50, 60, 70, 80])  # Same tokens but at block 1
print(f"[50,60,70,80] as block 0: hash={hash(tokens_at_pos0)}")
print(f"[50,60,70,80] as block 1: hash={hash(tokens_at_pos1)} (includes prefix)")
print(f"Hashes are different: {hash(tokens_at_pos0) != hash(tokens_at_pos1)}")

## 4. Eviction Policies for Cached Blocks

### LRU Eviction in vLLM

When the block pool is full and a new block is needed, vLLM evicts the **least recently used** cached block:

```python
# From vllm/core/evictor.py
class LRUEvictor:
    """Evicts blocks in LRU order."""
    
    def __init__(self):
        self.free_table: OrderedDict[int, BlockMetaData] = OrderedDict()
    
    def evict(self) -> Tuple[int, BlockMetaData]:
        """Evict the least recently used block."""
        if not self.free_table:
            raise ValueError("No blocks to evict")
        
        # Pop from the front (oldest)
        block_id, block_metadata = self.free_table.popitem(last=False)
        return block_id, block_metadata
    
    def add(self, block_id: int, block_metadata: BlockMetaData):
        """Add a block to the evictor (append to end = most recent)."""
        self.free_table[block_id] = block_metadata
    
    def remove(self, block_id: int):
        """Remove a block from the evictor (cache hit)."""
        if block_id in self.free_table:
            del self.free_table[block_id]
```

### Block Lifecycle

```
                ┌──────────┐
        ┌──────►│  FREE    │◄──────────── evict (remove hash)
        │       └────┬─────┘
        │            │ allocate (cache miss)
        │            ▼
        │       ┌──────────┐
        │       │  ACTIVE  │ ← ref_count > 0
        │       └────┬─────┘
        │            │ free (ref_count → 0)
        │            ▼
        │       ┌──────────┐
        └───────│  CACHED  │ ← ref_count = 0, computed, hash preserved
                │  (LRU)   │
                └────┬─────┘
                     │ cache hit (ref_count → 1)
                     └──────────► ACTIVE
```

In [ ]:
class LRUEvictor:
    """
    LRU eviction policy for cached blocks.
    Mirrors vllm/core/evictor.py
    """
    
    def __init__(self):
        self.cache: OrderedDict[int, CachedBlock] = OrderedDict()
        self.eviction_count = 0
    
    def add(self, block: CachedBlock):
        """Add block to cache (most recent position)."""
        self.cache[block.block_id] = block
        self.cache.move_to_end(block.block_id)  # Most recently used
    
    def access(self, block_id: int):
        """Mark block as recently used (move to end)."""
        if block_id in self.cache:
            self.cache.move_to_end(block_id)
    
    def remove(self, block_id: int) -> Optional[CachedBlock]:
        """Remove specific block (cache hit)."""
        if block_id in self.cache:
            return self.cache.pop(block_id)
        return None
    
    def evict(self) -> Optional[CachedBlock]:
        """Evict least recently used block."""
        if not self.cache:
            return None
        block_id, block = self.cache.popitem(last=False)  # Pop oldest
        self.eviction_count += 1
        return block
    
    def __len__(self):
        return len(self.cache)
    
    def __contains__(self, block_id: int):
        return block_id in self.cache


# Demonstrate LRU behavior
evictor = LRUEvictor()

# Add blocks in order
for i in range(5):
    block = CachedBlock(block_id=i, content_hash=hash(f"content_{i}"))
    evictor.add(block)

print("LRU order (oldest first):")
print(f"  {[bid for bid in evictor.cache.keys()]}")

# Access block 1 (moves to most recent)
evictor.access(1)
print(f"\nAfter accessing block 1:")
print(f"  {[bid for bid in evictor.cache.keys()]}")

# Evict (should evict block 0, the oldest)
evicted = evictor.evict()
print(f"\nEvicted block {evicted.block_id}")
print(f"Remaining: {[bid for bid in evictor.cache.keys()]}")

evicted = evictor.evict()
print(f"Evicted block {evicted.block_id}")
print(f"Remaining: {[bid for bid in evictor.cache.keys()]}")

## 5. APC Integration with the Scheduler

### How APC Interacts with vLLM's Scheduler

```
                    ┌──────────────┐
                    │  New Request  │
                    └──────┬───────┘
                           │
                           ▼
                 ┌─────────────────────┐
                 │  Compute block      │
                 │  hashes for all     │
                 │  token blocks       │
                 └─────────┬───────────┘
                           │
                           ▼
              ┌────────────────────────────┐
              │  For each block:            │
              │  Check hash_to_block table  │
              └────────────┬───────────────┘
                           │
              ┌────────────┴───────────────┐
              │                             │
              ▼                             ▼
      ┌──────────────┐            ┌──────────────┐
      │  Cache HIT   │            │  Cache MISS  │
      │  Reuse block │            │  Allocate    │
      │  Skip prefill│            │  new block   │
      └──────────────┘            │  Run prefill │
                                  └──────────────┘
                           │
                           ▼
                 ┌─────────────────────┐
                 │  Prefill only       │
                 │  non-cached tokens  │
                 │  (huge speedup!)    │
                 └─────────────────────┘
```

### Source Code: Scheduler Integration

```python
# From vllm/core/scheduler.py (simplified)
def _schedule_prefills(self):
    for seq_group in waiting_queue:
        # Compute number of blocks that can be reused from cache
        num_cached_blocks = self.block_manager.get_num_cached_blocks(seq_group)
        
        # Only need to prefill the non-cached portion
        num_tokens_to_compute = (
            seq_group.get_seqs()[0].get_len() - 
            num_cached_blocks * self.cache_config.block_size
        )
        
        # This dramatically reduces prefill cost!
```

In [ ]:
class APCServingSimulator:
    """
    Full simulation of APC-enabled serving.
    
    Simulates request arrival, prefix matching, and cache management.
    """
    
    def __init__(self, num_blocks: int = 500, block_size: int = 16):
        self.allocator = PrefixCachingBlockAllocator(num_blocks, block_size)
        self.block_size = block_size
        self.active_requests: Dict[int, List[CachedBlock]] = {}
        self.next_req_id = 0
        
        # Metrics
        self.total_tokens_processed = 0
        self.tokens_saved_by_cache = 0
        self.request_latencies = []
    
    def submit_request(self, token_ids: Tuple[int, ...]) -> dict:
        """
        Submit a new request and allocate blocks.
        Returns metrics about cache utilization.
        """
        req_id = self.next_req_id
        self.next_req_id += 1
        
        hashes = ContentHash.compute_all_hashes(token_ids, self.block_size)
        blocks = []
        cached_tokens = 0
        computed_tokens = 0
        
        for blk_idx, h in enumerate(hashes):
            block = self.allocator.allocate_block(content_hash=h)
            if block is None:
                # Out of blocks, free what we allocated
                for b in blocks:
                    self.allocator.free_block(b)
                return {'success': False, 'req_id': req_id}
            
            tokens_in_block = min(self.block_size, 
                                  len(token_ids) - blk_idx * self.block_size)
            
            if block.is_computed:
                cached_tokens += tokens_in_block
            else:
                computed_tokens += tokens_in_block
                self.allocator.mark_computed(block)
            
            blocks.append(block)
        
        self.active_requests[req_id] = blocks
        self.total_tokens_processed += len(token_ids)
        self.tokens_saved_by_cache += cached_tokens
        
        return {
            'success': True,
            'req_id': req_id,
            'total_tokens': len(token_ids),
            'cached_tokens': cached_tokens,
            'computed_tokens': computed_tokens,
            'num_blocks': len(blocks),
        }
    
    def complete_request(self, req_id: int):
        """Complete a request, releasing block references."""
        if req_id in self.active_requests:
            for block in self.active_requests[req_id]:
                self.allocator.free_block(block)
            del self.active_requests[req_id]

print("APCServingSimulator defined.")

In [ ]:
# Simulate a multi-turn conversation scenario

sim = APCServingSimulator(num_blocks=200, block_size=16)

# System prompt (shared by all users)
system_prompt = tuple(range(1000, 1064))  # 64 tokens = 4 blocks

# Simulate 5 users, each having 3-turn conversations
print("=== Multi-Turn Conversation Simulation ===")
print(f"System prompt: {len(system_prompt)} tokens\n")

conversations = {}
for user_id in range(5):
    conversations[user_id] = list(system_prompt)
    
    for turn in range(3):
        # Add user query (16-48 tokens)
        query_len = random.randint(16, 48)
        user_query = list(range(user_id * 10000 + turn * 1000, 
                                user_id * 10000 + turn * 1000 + query_len))
        conversations[user_id].extend(user_query)
        
        # Submit request
        token_ids = tuple(conversations[user_id])
        result = sim.submit_request(token_ids)
        
        if result['success']:
            cache_pct = result['cached_tokens'] / result['total_tokens'] * 100
            print(f"User {user_id}, Turn {turn}: "
                  f"{result['total_tokens']} tokens, "
                  f"{result['cached_tokens']} cached ({cache_pct:.0f}%), "
                  f"{result['computed_tokens']} computed")
            
            # Complete the request (simulating response generation)
            sim.complete_request(result['req_id'])
            
            # Add assistant response to conversation
            response_len = random.randint(32, 64)
            response = list(range(user_id * 10000 + turn * 1000 + 500,
                                  user_id * 10000 + turn * 1000 + 500 + response_len))
            conversations[user_id].extend(response)

print(f"\n=== Overall Stats ===")
print(f"Total tokens processed: {sim.total_tokens_processed:,}")
print(f"Tokens saved by cache: {sim.tokens_saved_by_cache:,}")
print(f"Computation saved: {sim.tokens_saved_by_cache/sim.total_tokens_processed:.1%}")
print(f"Cache hit rate: {sim.allocator.hit_rate:.1%}")
print(f"Evictions: {sim.allocator.evictions}")

## 6. Cache Hit Rate Analysis

In [ ]:
def analyze_hit_rates(workload_type: str, num_requests: int = 200, 
                       num_blocks: int = 500, seed: int = 42):
    """
    Analyze cache hit rates for different workload types.
    """
    random.seed(seed)
    sim = APCServingSimulator(num_blocks=num_blocks, block_size=16)
    
    hit_rate_history = []
    compute_savings = []
    
    if workload_type == 'chatbot':
        # All requests share same system prompt
        system_prompt = tuple(range(1000, 1128))  # 128 tokens
        for i in range(num_requests):
            query = tuple(range(i * 100, i * 100 + random.randint(16, 64)))
            tokens = system_prompt + query
            result = sim.submit_request(tokens)
            if result['success']:
                sim.complete_request(result['req_id'])
                hit_rate_history.append(sim.allocator.hit_rate)
                compute_savings.append(result['cached_tokens'] / result['total_tokens'])
    
    elif workload_type == 'few_shot':
        # 3 different few-shot templates, each used by ~1/3 of requests
        templates = [
            tuple(range(2000, 2256)),  # 256 tokens each
            tuple(range(3000, 3256)),
            tuple(range(4000, 4256)),
        ]
        for i in range(num_requests):
            template = templates[i % 3]
            query = tuple(range(i * 100 + 50000, i * 100 + 50000 + random.randint(16, 32)))
            tokens = template + query
            result = sim.submit_request(tokens)
            if result['success']:
                sim.complete_request(result['req_id'])
                hit_rate_history.append(sim.allocator.hit_rate)
                compute_savings.append(result['cached_tokens'] / result['total_tokens'])
    
    elif workload_type == 'unique':
        # No shared prefixes
        for i in range(num_requests):
            tokens = tuple(range(i * 1000, i * 1000 + random.randint(64, 256)))
            result = sim.submit_request(tokens)
            if result['success']:
                sim.complete_request(result['req_id'])
                hit_rate_history.append(sim.allocator.hit_rate)
                compute_savings.append(result['cached_tokens'] / result['total_tokens'])
    
    elif workload_type == 'multiturn':
        # Multi-turn conversations
        system_prompt = tuple(range(1000, 1064))  # 64 tokens
        conversations = {}
        num_users = 20
        
        for i in range(num_requests):
            user = i % num_users
            if user not in conversations:
                conversations[user] = list(system_prompt)
            
            # Add new query
            query = list(range(user * 10000 + len(conversations[user]),
                               user * 10000 + len(conversations[user]) + random.randint(16, 32)))
            conversations[user].extend(query)
            
            tokens = tuple(conversations[user])
            result = sim.submit_request(tokens)
            if result['success']:
                sim.complete_request(result['req_id'])
                hit_rate_history.append(sim.allocator.hit_rate)
                compute_savings.append(result['cached_tokens'] / result['total_tokens'])
                
                # Add response
                response = list(range(user * 10000 + 5000 + len(conversations[user]),
                                      user * 10000 + 5000 + len(conversations[user]) + random.randint(32, 64)))
                conversations[user].extend(response)
    
    return hit_rate_history, compute_savings

# Run all workloads
workloads = ['chatbot', 'few_shot', 'multiturn', 'unique']
results = {}
for wl in workloads:
    results[wl] = analyze_hit_rates(wl)
    final_hr = results[wl][0][-1] if results[wl][0] else 0
    avg_savings = np.mean(results[wl][1]) if results[wl][1] else 0
    print(f"{wl:>12}: final hit rate = {final_hr:.1%}, avg compute savings = {avg_savings:.1%}")

In [ ]:
# Plot hit rate and compute savings
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Hit rate over time
for wl in workloads:
    if results[wl][0]:
        axes[0].plot(results[wl][0], label=wl, alpha=0.8)
axes[0].set_xlabel('Request Number', fontsize=12)
axes[0].set_ylabel('Cumulative Hit Rate', fontsize=12)
axes[0].set_title('Cache Hit Rate Over Time', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

# Compute savings distribution
box_data = [results[wl][1] for wl in workloads]
bp = axes[1].boxplot(box_data, labels=workloads, patch_artist=True)
colors_box = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Compute Savings per Request', fontsize=12)
axes[1].set_title('Computation Saved by APC', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/apc_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. APC Implementation Details in vLLM Source Code

### Key Classes and Their Roles

```
vllm/core/block/prefix_caching_block.py:
├── PrefixCachingBlockAllocator
│   ├── _hashless_allocator: NaiveBlockAllocator  # For mutable (non-full) blocks
│   ├── _cached_blocks: Dict[int, int]            # hash -> block_id
│   └── evictor: LRUEvictor                       # Eviction policy
│
├── PrefixCachingBlock
│   ├── _block_id: int
│   ├── _content_hash: Optional[int]
│   ├── _prev_block: Optional[PrefixCachingBlock]
│   └── _cached: bool                              # Whether block has valid KV
│
└── Key Methods:
    ├── allocate_immutable_block()    # Allocate full block (can match cache)
    ├── allocate_mutable_block()      # Allocate partial block (no caching)
    ├── _promote_to_immutable()       # Convert mutable → immutable when full
    └── _get_cached_block()           # Look up block by content hash
```

### Immutable vs Mutable Blocks

```
Immutable Block (Full, can be cached):
  ┌──────────────────────┐
  │ [tok0][tok1]...[tok15]│ ← All 16 slots filled
  │ hash = 0xABCD1234     │ ← Content hash computed
  │ cacheable = True      │ ← Can be reused by other sequences
  └──────────────────────┘

Mutable Block (Partial, cannot be cached):
  ┌──────────────────────┐
  │ [tok0][tok1][___][___]│ ← Only 2 of 16 slots filled
  │ hash = None           │ ← No hash (content may change)
  │ cacheable = False     │ ← Cannot be reused
  └──────────────────────┘
```

In [ ]:
class PrefixCachingBlockV2:
    """
    More detailed simulation of vLLM's PrefixCachingBlock.
    
    Distinguishes between mutable and immutable blocks.
    """
    
    def __init__(self, block_id: int, block_size: int, 
                 prev_block=None, token_ids=None):
        self.block_id = block_id
        self.block_size = block_size
        self.prev_block = prev_block
        self.token_ids = list(token_ids) if token_ids else []
        self._content_hash = None
        self.ref_count = 1
        self.is_computed = False
        
        if self.is_full:
            self._compute_hash()
    
    @property
    def is_full(self) -> bool:
        return len(self.token_ids) >= self.block_size
    
    @property
    def is_immutable(self) -> bool:
        return self.is_full
    
    @property
    def content_hash(self) -> Optional[int]:
        return self._content_hash
    
    def _compute_hash(self):
        """Compute hash including all preceding tokens."""
        all_tokens = []
        block = self
        while block is not None:
            all_tokens = block.token_ids + all_tokens
            block = block.prev_block
        self._content_hash = hash(tuple(all_tokens[:self.block_size * 
                                                     (len(all_tokens) // self.block_size)]))
    
    def append_token(self, token_id: int):
        """Append a token to a mutable block."""
        if self.is_full:
            raise RuntimeError("Cannot append to full block")
        self.token_ids.append(token_id)
        if self.is_full:
            self._compute_hash()
    
    def __repr__(self):
        state = "IMMUTABLE" if self.is_immutable else "MUTABLE"
        computed = "computed" if self.is_computed else "pending"
        return (f"Block(id={self.block_id}, {state}, "
                f"tokens={len(self.token_ids)}/{self.block_size}, "
                f"{computed}, refs={self.ref_count})")


# Demonstrate mutable → immutable transition
block_size = 4
block = PrefixCachingBlockV2(block_id=0, block_size=block_size)

print("Mutable → Immutable Transition:")
for token in [10, 20, 30, 40]:
    block.append_token(token)
    print(f"  After adding token {token}: {block}")
    print(f"    Hash: {block.content_hash}")

print(f"\nBlock is now immutable and can be cached!")

## 8. Performance Impact Measurement

In [ ]:
def estimate_ttft_improvement(prompt_len: int, cached_tokens: int, 
                                tokens_per_second: int = 10000):
    """
    Estimate Time-To-First-Token improvement from APC.
    
    TTFT is dominated by prefill computation.
    With APC, we skip computation for cached tokens.
    """
    ttft_without_cache = prompt_len / tokens_per_second
    ttft_with_cache = (prompt_len - cached_tokens) / tokens_per_second
    speedup = ttft_without_cache / max(ttft_with_cache, 0.0001)
    
    return {
        'ttft_no_cache_ms': ttft_without_cache * 1000,
        'ttft_with_cache_ms': ttft_with_cache * 1000,
        'speedup': speedup,
        'saved_ms': (ttft_without_cache - ttft_with_cache) * 1000,
    }

# Scenarios
scenarios = [
    ('Short chatbot', 200, 128),       # System prompt cached
    ('Long chatbot', 500, 384),        # System prompt + examples
    ('Multi-turn (turn 5)', 2000, 1800), # Previous turns cached
    ('RAG with context', 4000, 3500),  # Retrieved docs cached
    ('Code completion', 8000, 7500),   # Large file context cached
]

print(f"{'Scenario':<25} {'Prompt':>7} {'Cached':>7} {'No Cache':>10} {'With Cache':>11} {'Speedup':>8}")
print("-" * 72)

for name, prompt_len, cached in scenarios:
    r = estimate_ttft_improvement(prompt_len, cached)
    print(f"{name:<25} {prompt_len:>7} {cached:>7} "
          f"{r['ttft_no_cache_ms']:>9.1f}ms {r['ttft_with_cache_ms']:>10.1f}ms "
          f"{r['speedup']:>7.1f}x")

In [ ]:
# Visualize TTFT improvement
fig, ax = plt.subplots(figsize=(12, 6))

names = [s[0] for s in scenarios]
no_cache = [estimate_ttft_improvement(s[1], s[2])['ttft_no_cache_ms'] for s in scenarios]
with_cache = [estimate_ttft_improvement(s[1], s[2])['ttft_with_cache_ms'] for s in scenarios]

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, no_cache, width, label='Without APC', color='#F44336', alpha=0.8)
bars2 = ax.bar(x + width/2, with_cache, width, label='With APC', color='#4CAF50', alpha=0.8)

ax.set_ylabel('Time to First Token (ms)', fontsize=12)
ax.set_title('TTFT Improvement with Automatic Prefix Caching', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

# Add speedup annotations
for i, s in enumerate(scenarios):
    r = estimate_ttft_improvement(s[1], s[2])
    ax.annotate(f"{r['speedup']:.1f}x", 
                xy=(i, max(no_cache[i], with_cache[i])),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/ttft_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cache Size vs Hit Rate Tradeoff

In [ ]:
# Analyze how cache size affects hit rate

cache_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
workload = 'chatbot'

hit_rates = []
eviction_counts = []

for num_blocks in cache_sizes:
    hr, cs = analyze_hit_rates(workload, num_requests=200, num_blocks=num_blocks)
    final_hr = hr[-1] if hr else 0
    hit_rates.append(final_hr)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(cache_sizes, hit_rates, 'bo-', linewidth=2, markersize=8)
ax.set_xscale('log')
ax.set_xlabel('Number of Cache Blocks', fontsize=12)
ax.set_ylabel('Cache Hit Rate', fontsize=12)
ax.set_title('Cache Hit Rate vs Cache Size (Chatbot Workload)', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Annotate
for i, (size, hr) in enumerate(zip(cache_sizes, hit_rates)):
    ax.annotate(f'{hr:.1%}', (size, hr), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/cache_size_vs_hit_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Enabling APC in vLLM

### Configuration

```python
# Enable APC via command line:
# python -m vllm.entrypoints.openai.api_server \
#     --model meta-llama/Llama-2-7b-chat-hf \
#     --enable-prefix-caching

# Or via Python API:
from vllm import LLM, SamplingParams

llm = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    enable_prefix_caching=True,  # Enable APC
    # Optional: adjust block size
    # block_size=16,  # default
)

# APC is transparent - no changes to application code needed!
# The system automatically detects shared prefixes.
```

### When to Enable APC

| Use Case | Benefit | Recommendation |
|----------|---------|----------------|
| Chatbot with system prompt | High | Always enable |
| Multi-turn conversations | Very High | Always enable |
| Few-shot prompting | High | Always enable |
| RAG with cached docs | Medium-High | Enable if docs are reused |
| Unique prompts | Low | Slight overhead, but no harm |
| Benchmark/testing | None | Can disable for reproducibility |

## 11. Advanced: Prefix Tree Visualization

In [ ]:
class PrefixTree:
    """Visualize how prefix caching creates a tree of shared blocks."""
    
    def __init__(self, block_size: int = 4):
        self.block_size = block_size
        self.nodes: Dict[int, dict] = {}  # hash -> {tokens, children, count}
        self.root_hashes: Set[int] = set()
    
    def add_sequence(self, token_ids: Tuple[int, ...]):
        """Add a sequence to the prefix tree."""
        hashes = ContentHash.compute_all_hashes(token_ids, self.block_size)
        prev_hash = None
        
        for blk_idx, h in enumerate(hashes):
            if h is None:
                break
            
            start = blk_idx * self.block_size
            end = start + self.block_size
            block_tokens = token_ids[start:end]
            
            if h not in self.nodes:
                self.nodes[h] = {
                    'tokens': block_tokens,
                    'children': set(),
                    'count': 0,
                    'parent': prev_hash,
                    'depth': blk_idx,
                }
            
            self.nodes[h]['count'] += 1
            
            if prev_hash is not None and prev_hash in self.nodes:
                self.nodes[prev_hash]['children'].add(h)
            elif prev_hash is None:
                self.root_hashes.add(h)
            
            prev_hash = h
    
    def print_tree(self, max_depth: int = 5):
        """Print the prefix tree."""
        def _print(h, indent=0):
            if indent // 2 >= max_depth:
                return
            node = self.nodes[h]
            tokens_str = str(tuple(node['tokens']))
            print(f"{'  ' * indent}[{tokens_str}] (refs={node['count']})")
            for child_h in sorted(node['children']):
                _print(child_h, indent + 1)
        
        print("Prefix Tree:")
        for root_h in sorted(self.root_hashes):
            _print(root_h)

# Build prefix tree
tree = PrefixTree(block_size=4)

# System prompt
system = (1, 2, 3, 4, 5, 6, 7, 8)

# Different queries with same system prompt
sequences = [
    system + (10, 11, 12, 13),   # User A
    system + (10, 11, 12, 13, 20, 21, 22, 23),  # User A, longer
    system + (30, 31, 32, 33),   # User B
    system + (30, 31, 32, 33, 40, 41, 42, 43),  # User B, longer
    system + (50, 51, 52, 53),   # User C
]

for seq in sequences:
    tree.add_sequence(seq)

tree.print_tree()

print(f"\nTotal unique blocks: {len(tree.nodes)}")
print(f"Total block references: {sum(n['count'] for n in tree.nodes.values())}")
print(f"Sharing ratio: {sum(n['count'] for n in tree.nodes.values()) / len(tree.nodes):.1f}x")

## 12. Summary

### Key Concepts

| Concept | Description |
|---------|-------------|
| Content Hash | Blocks identified by hash of all preceding tokens |
| Cache Hit | Reuse existing computed block instead of recomputing |
| LRU Eviction | When cache is full, evict least recently used block |
| Immutable Block | Full block that can be cached and shared |
| Mutable Block | Partial block, not cacheable |
| Hash Chain | Each block's hash depends on full prefix history |

### Performance Impact
- **10-40x TTFT improvement** for prefix-heavy workloads
- **Near-zero overhead** for workloads without shared prefixes
- **Automatic** — no application code changes needed

### Best Practices
1. Always enable APC for production chatbot deployments
2. Use consistent system prompts across requests
3. Structure multi-turn conversations with stable prefixes
4. Monitor cache hit rate to validate benefits

## Exercises

### Exercise 1: Implement a Custom Eviction Policy
Implement a frequency-based eviction policy (LFU) and compare it with LRU for different workloads.

### Exercise 2: Design an Optimal Caching Strategy
Given a workload with:
- 5 system prompts (each 256 tokens), used with frequencies 40%, 25%, 20%, 10%, 5%
- Average user query: 64 tokens
- Budget: 100 blocks of size 16

Design a caching strategy that maximizes hit rate.

### Exercise 3: Prefix Caching with Beam Search
Analyze how prefix caching interacts with beam search. When beams share a common prefix that is already cached, how much memory is saved compared to no caching?

### Exercise 4: Cache Warming
Implement a cache warming strategy that pre-populates the cache with known system prompts at startup.

In [ ]:
# Exercise 1 Starter: LFU Evictor

class LFUEvictor:
    """Least Frequently Used eviction policy."""
    
    def __init__(self):
        self.cache: Dict[int, CachedBlock] = {}
        self.access_counts: Dict[int, int] = {}
    
    def add(self, block: CachedBlock):
        self.cache[block.block_id] = block
        self.access_counts[block.block_id] = self.access_counts.get(block.block_id, 0)
    
    def access(self, block_id: int):
        if block_id in self.access_counts:
            self.access_counts[block_id] += 1
    
    def evict(self) -> Optional[CachedBlock]:
        if not self.cache:
            return None
        # Find block with minimum access count
        min_block_id = min(self.cache.keys(), 
                           key=lambda bid: self.access_counts.get(bid, 0))
        block = self.cache.pop(min_block_id)
        del self.access_counts[min_block_id]
        return block
    
    def __len__(self):
        return len(self.cache)

# TODO: Integrate LFUEvictor into PrefixCachingBlockAllocator
# and compare with LRU on different workloads
print("LFUEvictor defined. Integrate it with the allocator and compare!")

In [ ]:
# Exercise 4 Starter: Cache Warming

def warm_cache(simulator: APCServingSimulator, 
               system_prompts: List[Tuple[int, ...]]):
    """
    Pre-populate the prefix cache with known system prompts.
    
    Strategy: Submit and immediately complete requests with each prompt.
    This leaves the prompt blocks in the cache for future reuse.
    """
    print("Warming cache with system prompts...")
    for i, prompt in enumerate(system_prompts):
        result = simulator.submit_request(prompt)
        if result['success']:
            simulator.complete_request(result['req_id'])
            print(f"  Prompt {i}: {len(prompt)} tokens cached")
    print(f"Cache warmed with {len(system_prompts)} prompts.")
    print(f"Cached blocks: {simulator.allocator.stats['cached_blocks']}")

# Demo
sim = APCServingSimulator(num_blocks=500, block_size=16)

prompts = [
    tuple(range(1000, 1128)),  # 128 tokens
    tuple(range(2000, 2256)),  # 256 tokens
    tuple(range(3000, 3064)),  # 64 tokens
]

warm_cache(sim, prompts)

# Now subsequent requests with these prompts will get cache hits
test_tokens = prompts[0] + tuple(range(5000, 5032))  # Prompt 0 + new query
result = sim.submit_request(test_tokens)
print(f"\nTest request: {result['cached_tokens']}/{result['total_tokens']} tokens from cache")